# Exercice 3.1 : un quatrième spécialiste, Accès (40 min)

> Prérequis : J3-02 (KB indexée). Notebook **autonome**, asynchrone.

**Contexte :** un tiers des demandes du help-desk de NordTech concernent les **comptes** (verrouillés, expirés, désactivés). Le coordinateur doit disposer d'un spécialiste Accès, sans rien casser des trois autres.

| Étape | Consigne |
|---|---|
| 1. Outil annuaire | `etat_compte(identifiant)` lit `COMPTES` ; docstring précise, exemple d'identifiant |
| 2. Outil procédures | `procedures_acces(question)` : comme `rechercher_procedures`, mais **filtré sur la catégorie `acces`** (`kb.rechercher(…, categorie="acces")`) |
| 3. Spécialiste | `create_agent` avec ces deux outils et une consigne : vérifier le compte, citer la procédure `[KB-xxx]`, ne **jamais** réinitialiser soi-même |
| 4. Coordinateur | `construire_coordinateur(llm, specialistes, descriptions)` avec le 4ᵉ spécialiste et **sa description** |
| 5. Tests | les 3 questions fournies, session `ex31-<initiales>` |
| 6. Langfuse | pour chaque trace : quels spécialistes ont été appelés ? avec quelle consigne ? |

**Critères de réussite :** la question 1 appelle `demander_acces` (et lui seul) · la question 2 ne l'appelle **pas** · la question 3 appelle Accès **et** Diagnostic · les réponses citent une procédure `[KB-xxx]` de la catégorie accès · aucun nom d'utilisateur n'apparaît dans les consignes des spécialistes.

> 💡 **Le « Tips » du Formateur :** c'est la **description** de l'outil `demander_acces` qui décide du routage, pas son code. Essayez une description vague (« gère les accès ») puis une description précise : comparez les traces.

## 0. Mise en route et éléments fournis

In [ ]:
import os

from langfuse import observe, propagate_attributes

from helpdesk_ops.config import creer_llm
from helpdesk_ops.observabilite import init_observabilite
from helpdesk_ops.texte import sans_accents

langfuse, langfuse_handler = init_observabilite()   # .env chargé, masquage PII actif, clés vérifiées
llm = creer_llm("agent")                             # MODEL_AGENT, temperature=0, num_ctx=16384
print(f"✅ Langfuse : {os.environ['LANGFUSE_BASE_URL']} | modèle : {llm.model}")

In [ ]:
import time

from langchain.agents import create_agent
from langchain_core.tools import tool

from helpdesk_ops import kb
from helpdesk_ops.multi_agents import construire_coordinateur, construire_specialistes

specialistes = construire_specialistes(llm)   # kb, diagnostic, ticketing (J3-02)

# Annuaire simulé de NordTech (identifiants pseudonymisés)
COMPTES = {
    "collab-3fa91c": "actif",
    "collab-8b20d4": "verrouillé (3 échecs de connexion, déverrouillage automatique dans 15 min)",
    "collab-71e0aa": "mot de passe expiré",
    "collab-c45f12": "désactivé (départ le 2026-09-30)",
}

QUESTIONS = [
    "Mon compte collab-8b20d4 est bloqué, que faire ?",                        # → acces
    "Outlook plante dès que j'ouvre une pièce jointe",                          # → kb (PAS acces)
    "Le compte collab-71e0aa ne marche plus, et srv-erp-02 répond-il au ping ?",  # → acces ET diagnostic
]


async def poser(coordinateur, question: str, session: str) -> None:
    debut = time.perf_counter()
    with propagate_attributes(session_id=session, tags=["jour-3", "ex31"]):
        etat = await coordinateur.ainvoke({"messages": [("user", question)]},
                                          config={"callbacks": [langfuse_handler], "run_name": "ex31", "recursion_limit": 25})
    appels = [c["name"] for m in etat["messages"] for c in (getattr(m, "tool_calls", None) or [])]
    print(f"⏱ {time.perf_counter() - debut:4.0f} s | {appels}\n❓ {question}\n🤖 {etat['messages'][-1].content[:300]}\n")

## 1. Les outils du spécialiste

In [ ]:
@tool(parse_docstring=True)
def etat_compte(identifiant: str) -> str:
    """TODO

    Args:
        identifiant: TODO
    """
    ...  # TODO


@tool(parse_docstring=True)
def procedures_acces(question: str) -> str:
    """TODO

    Args:
        question: TODO
    """
    ...  # TODO : kb.formater(kb.rechercher(question, k=3, categorie="acces"))

## 2. Le spécialiste et le coordinateur

In [ ]:
specialiste_acces = ...   # TODO : create_agent(llm, [...], system_prompt=..., name="specialiste_acces")

DESCRIPTION_ACCES = "TODO : quand le coordinateur doit-il appeler ce spécialiste ?"
coordinateur = ...        # TODO : construire_coordinateur(llm, {**specialistes, "acces": ...}, {"acces": DESCRIPTION_ACCES})

## 3. Les tests

In [ ]:
INITIALES = "xx"   # TODO
for q in QUESTIONS:
    await poser(coordinateur, q, f"ex31-{INITIALES}")
langfuse.flush()

## 4. Dans Langfuse

1. Ouvrez la trace de la question 3 : dans quel ordre les spécialistes sont-ils appelés ? En parallèle ou l'un après l'autre ?
2. Lisez les consignes envoyées aux spécialistes : sont-elles autoportantes (un humain comprendrait-il la tâche sans la conversation) ?
3. Combien d'appels LLM au total pour la question 3 ? Qu'en concluez-vous sur le coût d'un spécialiste de plus ?